# Unique pages referenced by face-entry page span

This notebook counts unique Economist page identifiers referenced by all deduplicated 1940–2007 face entries, grouped by the number of encoded source pages in each entry. Face rows are first collapsed to unique source scans, because one source scan can contain several detected faces.

It reports each observed page-span length, plus the combined sets for one- or two-page entries (`1+2`) and entries spanning three or more pages (`3+`).

## Inputs and output

The cleaned CSV supplies one- and two-page scans, while the companion dropped-entry CSV supplies scans spanning three or more pages. Combining them reconstructs the complete deduplicated 1940–2007 set needed for the complementary `3+` total.

In [ ]:
from pathlib import Path

import pandas as pd

CLEANED_CSV = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_cleaned.csv")
DROPPED_CSV = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated_dropped-over-two-pages.csv")
OUTPUT_CSV = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-unique-pages-by-entry-span.csv")

for path in (CLEANED_CSV, DROPPED_CSV):
    assert path.exists(), f"Missing input CSV: {path}"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f"Cleaned input: {CLEANED_CSV}")
print(f"Dropped input: {DROPPED_CSV}")
print(f"Output: {OUTPUT_CSV}")

## Parse and validate source scans

The filename prefix has the form `yyyy-mmdd-pppp[,pppp...]`. Its comma-separated page list is the source of the span length and the page identifiers counted below.

In [ ]:
FILENAME_PATTERN = r"^(?P<issue_id>\d{4}-\d{4})-(?P<source_pages>\d{4}(?:,\d{4})*)_"

cleaned_faces = pd.read_csv(CLEANED_CSV, usecols=["Filename"], dtype={"Filename": "string"})
dropped_faces = pd.read_csv(DROPPED_CSV, usecols=["Filename"], dtype={"Filename": "string"})
assert len(cleaned_faces) > 0 and len(dropped_faces) > 0

scan_frames = []
for source_name, faces in [("cleaned", cleaned_faces), ("dropped_over_two_pages", dropped_faces)]:
    scan_parts = faces["Filename"].str.extract(FILENAME_PATTERN)
    assert scan_parts.notna().all().all(), f"Unparseable Filename values in {source_name}."
    scan_frames.append(scan_parts.assign(source_file=source_name))

source_scans = pd.concat(scan_frames, ignore_index=True).drop_duplicates().copy()
source_scans["source_scan_id"] = source_scans["issue_id"] + "-" + source_scans["source_pages"]
source_scans["page_numbers"] = source_scans["source_pages"].str.split(",")
source_scans["entry_page_count"] = source_scans["page_numbers"].str.len().astype("int64")

assert source_scans["source_scan_id"].is_unique
assert source_scans.loc[source_scans["source_file"].eq("cleaned"), "entry_page_count"].between(1, 2).all()
assert source_scans.loc[source_scans["source_file"].eq("dropped_over_two_pages"), "entry_page_count"].gt(2).all()

print({
    "cleaned_face_rows": len(cleaned_faces),
    "dropped_face_rows": len(dropped_faces),
    "unique_source_scans": len(source_scans),
    "span_lengths_present": sorted(source_scans["entry_page_count"].unique().tolist()),
})

## Count unique pages

A page is counted once within each aggregation even if it is referenced by more than one source scan. The source-scan count is included as context, but the primary result is `unique_page_count`.

In [ ]:
def referenced_page_ids(scans):
    return {
        f"{scan.issue_id}-{page_number}"
        for scan in scans.itertuples(index=False)
        for page_number in scan.page_numbers
    }

summary_rows = []
for page_count in sorted(source_scans["entry_page_count"].unique()):
    scans = source_scans.loc[source_scans["entry_page_count"].eq(page_count)]
    summary_rows.append({
        "entry_page_span": str(int(page_count)),
        "aggregation_rule": f"entry_page_count == {int(page_count)}",
        "source_scan_count": len(scans),
        "unique_page_count": len(referenced_page_ids(scans)),
    })

for label, rule, scans in [
    ("1+2", "entry_page_count in {1, 2}", source_scans.loc[source_scans["entry_page_count"].isin([1, 2])]),
    ("3+", "entry_page_count >= 3", source_scans.loc[source_scans["entry_page_count"].ge(3)]),
]:
    summary_rows.append({
        "entry_page_span": label,
        "aggregation_rule": rule,
        "source_scan_count": len(scans),
        "unique_page_count": len(referenced_page_ids(scans)),
    })

summary = pd.DataFrame(summary_rows)
assert summary["entry_page_span"].is_unique
summary

## Write and verify summary

The CSV provides a compact, reusable record of the page counts and aggregation rules.

In [ ]:
summary.to_csv(OUTPUT_CSV, index=False)
reloaded_summary = pd.read_csv(OUTPUT_CSV, dtype={"entry_page_span": "string"})

assert list(reloaded_summary.columns) == list(summary.columns)
assert reloaded_summary.equals(summary.astype({"entry_page_span": "string"}))

print(f"Wrote {len(reloaded_summary)} summary rows to {OUTPUT_CSV}")
reloaded_summary